In [ ]:
# !pip install folium pandas openpyxl
# !pip install pandas openpyxl plotly

import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster


In [ ]:
FILEPATH = "Database_Summary.xlsx"

df = pd.read_excel(FILEPATH, engine="openpyxl")

df["StartLat"] = pd.to_numeric(df["StartLat"], errors="coerce")
df["StartLon"] = pd.to_numeric(df["StartLon"], errors="coerce")

# remove missing and obviously invalid
dfp = df.dropna(subset=["StartLat", "StartLon"]).copy()
dfp = dfp[
    dfp["StartLat"].between(-90, 90) &
    dfp["StartLon"].between(-180, 180)
].copy()

# Optional: remove "0,0" points (common placeholder)
dfp = dfp[~((dfp["StartLat"].abs() < 1e-6) & (dfp["StartLon"].abs() < 1e-6))].copy()

# Kits of interest (as strings, keep leading zeros)
kits = ["01", "05", "06", "10", "11", "18", "24", "27", "30", "32"]

# Build regex pattern: match any of the numbers
pattern = "(" + "|".join(kits) + ")"

# Ensure Folder is string
dfp["Folder"] = dfp["Folder"].astype(str)
# Replace DatiXX -> KitXX
dfp["Folder"] = dfp["Folder"].str.replace(
    r"^Dati(\d+)$",   # capture the numeric part
    r"Kit\1",         # reuse the captured number
    regex=True
)
# Filter
dfp = dfp[dfp["Folder"].str.contains(pattern, regex=True)].copy()

dfp.shape


In [ ]:
import plotly.express as px

fig = px.scatter_map(
    dfp,
    lat="StartLat",
    lon="StartLon",
    color="Folder",        # legend toggles per kit
    hover_data={
        "Folder": True,
        "Wagon": True,
        "Filename": False,
        "Date": True,
        "StartTime": False,
        "IsMoving": False,
        "MaxSpeed": True,
        "TotalDistance_km": False,
        "StartLat": ":.5f",
        "StartLon": ":.5f",
    },
    height=750
)

fig.update_traces(
    marker=dict(size=8, opacity=0.85)
)

fig.update_layout(
    title="Instrumented Wagon Routes",
    title_x=0.5,                 # <-- horizontal center
    title_xanchor="center",      # <-- anchor at center
    legend_title_text="Monitoring Kit",
    map_style="open-street-map",
    margin=dict(l=0, r=0, t=40, b=0),
)

min_lat, max_lat = dfp["StartLat"].min(), dfp["StartLat"].max()
min_lon, max_lon = dfp["StartLon"].min(), dfp["StartLon"].max()

center_lat = float((min_lat + max_lat) / 2)
center_lon = float((min_lon + max_lon) / 2)

span = max(max_lat - min_lat, max_lon - min_lon)

if span < 0.05:
    zoom = 13
elif span < 0.2:
    zoom = 10
elif span < 1.0:
    zoom = 7
elif span < 5.0:
    zoom = 5
else:
    zoom = 4

fig.update_layout(
    map=dict(
        center=dict(lat=center_lat, lon=center_lon),
        zoom=5
    )
)
fig.update_layout(
    legend=dict(
        x=0.99,
        y=0.99,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="rgba(0,0,0,0.2)",
        borderwidth=1
    )
)

fig.show()


In [ ]:
fig.write_image(
    "monitoring_kits_map.png",
    width=768,   # pixels
    height=720,
    scale=1       # multiplies DPI (≈300 dpi)
)



In [ ]:
def remove_iqr_outliers(group, col="MaxSpeed", k=1.5):
    """
    Remove outliers from a group using IQR rule.
    """
    q1 = group[col].quantile(0.25)
    q3 = group[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - k * iqr
    upper = q3 + k * iqr

    return group[(group[col] >= lower) & (group[col] <= upper)]


In [ ]:
import plotly.express as px
# Ensure MaxSpeed is numeric
dfp["MaxSpeed"] = pd.to_numeric(dfp["MaxSpeed"], errors="coerce")

# Filter condition
df_highspeed = dfp[dfp["MaxSpeed"] > 20].copy()

df_highspeed_clean = (
    df_highspeed
    .groupby("Folder", group_keys=False)
    .apply(remove_iqr_outliers, col="MaxSpeed", k=1.5)
    .copy()
)

df_highspeed_clean.shape

summary = (
    df_highspeed
    .groupby("Folder")["MaxSpeed"].count()
    .rename("before")
    .to_frame()
)

summary["after"] = df_highspeed_clean.groupby("Folder")["MaxSpeed"].count()
summary["removed"] = summary["before"] - summary["after"]

summary.fillna(0).astype(int)



In [ ]:
import plotly.express as px
# Ensure MaxSpeed is numeric
dfp["MaxSpeedRPM"] = pd.to_numeric(dfp["MaxSpeedRPM"], errors="coerce")

# Filter condition
df_highspeed_rpm = dfp[dfp["MaxSpeedRPM"] > 20].copy()
# Numerical tolerance (important for floats)
tol = 1e-6
BAD_SPEED = 53.46412643361568
# Remove specific bad speed value with tolerance 
df_highspeed_rpm_clean = df_highspeed_rpm[
    ~df_highspeed_rpm["MaxSpeedRPM"].between(
        BAD_SPEED - tol,
        BAD_SPEED + tol
    )
].copy()

df_highspeed_rpm_clean.shape


In [ ]:
import plotly.express as px

kits_keep = ["Kit06", "Kit10", "Kit11", "Kit27"]

df_plot = df_highspeed_rpm_clean[
    df_highspeed_rpm_clean["Folder"].isin(kits_keep)
].copy()


fig_box = px.box(
    df_plot,
    x="Folder",
    y="MaxSpeedRPM",
    points="outliers",
    title="Operational Speed Distribution of the Instrumented Wagon"
)

fig_box.update_layout(
    title_x=0.5,
    boxmode="group",         # <-- side-by-side
    xaxis_title="Instrumented Wagon",
    yaxis_title="Measured Speed (km/h)",
    xaxis_tickangle=0,
    legend_title_text="Speed Source",
    margin=dict(l=20, r=20, t=60, b=40)
)
# --- hard y-axis limits ---
fig_box.update_yaxes(range=[0, 150])

fig_box.show()


In [ ]:
# --- GPS-based speed ---
df_gps = df_highspeed.copy()
df_gps["Speed"] = df_gps["MaxSpeed"]
df_gps["SpeedSource"] = "GPS"

# --- RPM-based speed ---
df_rpm = df_highspeed_rpm_clean.copy()
df_rpm["Speed"] = df_rpm["MaxSpeedRPM"]
df_rpm["SpeedSource"] = "Axle-box RPM"

# Keep only relevant columns
cols = ["Folder", "Speed", "SpeedSource"]

df_box = pd.concat(
    [df_gps[cols], df_rpm[cols]],
    ignore_index=True
)

df_box.head()


In [ ]:
import plotly.express as px

fig_box = px.box(
    df_box,
    x="Folder",
    y="Speed",
    color="SpeedSource",     # color coding
    points="outliers",
    title="Maximum Speed Distribution (GPS vs Axle-box RPM) of each Instrumented Wagon"
)

fig_box.update_layout(
    title_x=0.5,
    boxmode="group",         # <-- side-by-side
    xaxis_title="Instrumented Wagon Monitoring Kit",
    yaxis_title="Measured Speed (km/h)",
    xaxis_tickangle=0,
    legend_title_text="Speed Source",
    margin=dict(l=60, r=20, t=60, b=120)
)

fig_box.show()
